In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import warnings
warnings.filterwarnings("ignore") # Suppresses convergence warnings

In [ ]:
data = pd.read_csv(
    "daily_sales.csv",
    parse_dates=["date"],
    index_col="date"
)

data = data.sort_index()

values = data["sales"].values.astype(np.float32)

In [ ]:
# Check Sunday sales
sundays = data[data.index.weekday == 6]  # Sunday = 6

sundays[["sales"]]

In [ ]:
data.describe()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(data["sales"])
plt.title("Daily E-commerce Sales Values")
plt.xlabel("Date")
plt.ylabel("sales")
plt.grid(True)
plt.show()

In [ ]:
from statsmodels.tsa.stattools import adfuller

adf_test = adfuller(data["sales"])
# Output the results
print('ADF Statistic: %f' % adf_test[0])
print('p-value: %f' % adf_test[1])

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

# ETS Decomposition
result = seasonal_decompose(
    data["sales"],
    model="additive",
    period=6
)

result.plot()
plt.show()


In [ ]:
#keep the original data for later evaluation!
data2 = data.copy()

In [ ]:
model_results = []

In [ ]:
#Will use data for training, validation and testing up untill 27/6/2026
data = data.loc[:"2026-06-27"]

In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np
# Extract the values
values = data["sales"].values.astype(np.float32)

# Split into training validation and testing
n = len(values)
train_end = int(n * 0.70)
valid_end = int(n * 0.85)

train_series = values[:train_end]
valid_series = values[train_end:valid_end]
test_series = values[valid_end:]

# Print the length of the datasets to check
print("Number of training samples   : ",len(train_series))
print("Number of validation samples : ",len(valid_series))
print("Number of testing samples    : ",len(test_series))

# Scale the data
scaler = StandardScaler()

# Reshape is required for the scaler
train_series_reshaped = train_series.reshape(-1, 1)
scaler.fit(train_series_reshaped)

# Transform
train_series_scaled = scaler.transform(train_series.reshape(-1, 1)).flatten()
valid_series_scaled = scaler.transform(valid_series.reshape(-1, 1)).flatten()
test_series_scaled = scaler.transform(test_series.reshape(-1, 1)).flatten()

# Function to be used for inverse transformation
def inverse_transform(series, scaler):
    return scaler.inverse_transform(series.reshape(-1, 1)).reshape(series.shape)

In [ ]:
# Past and Future steps
#Monthly transactions are 26 days and we want to predict the next 6 days that is a typical week of sales.
past_steps = 26
future_steps = 6

# Function that makes windows of data to be used with the RNNs
def make_windows(series, past_steps, future_steps):
    X, Y = [], []
    for i in range(len(series) - past_steps - future_steps + 1):
        past = series[i:i + past_steps]
        future = series[i + past_steps:i + past_steps + future_steps]
        X.append(past)
        Y.append(future)
    X = np.array(X)[..., np.newaxis]
    Y = np.array(Y)[..., np.newaxis]
    return X, Y

# Make windows for all datasets
X_train, Y_train = make_windows(train_series_scaled, past_steps, future_steps)
X_valid, Y_valid = make_windows(valid_series_scaled, past_steps, future_steps)
X_test, Y_test = make_windows(test_series_scaled, past_steps, future_steps)

# Print the shapes to ensure they are conforming
print("Shape of Training data   : ", X_train.shape, Y_train.shape)
print("Shape of Validation data : ", X_valid.shape, Y_valid.shape)
print("Shape of Testing data    : ", X_test.shape, Y_test.shape)

# Visualize one training sample and its target
def plot_series(input_seq, target_seq=None, pred_seq=None):
    plt.figure(figsize=(10, 4))
    plt.plot(range(len(input_seq)), input_seq[:, 0], label="Input (past 26 days)")
    
    if target_seq is not None:
        plt.plot(
            range(len(input_seq), len(input_seq) + len(target_seq)),
            target_seq[:, 0],
            "bo-",
            label="Target (next 6 days)"
        )
    
    if pred_seq is not None:
        plt.plot(
            range(len(input_seq), len(input_seq) + len(pred_seq)),
            pred_seq[:, 0],
            "rx--",
            label="Prediction"
        )
    
    plt.axvline(len(input_seq) - 1, color="k", linestyle=":")
    plt.legend()
    plt.grid(True)
    plt.show()

plot_series(inverse_transform(X_train[0],scaler), inverse_transform(Y_train[0],scaler))


In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

# Number of epochs
epochs = 200

# Create default RNN
model_rnn = keras.models.Sequential([
    keras.layers.Input(shape=[past_steps, 1]),
    keras.layers.SimpleRNN(32, return_sequences=True),
    keras.layers.Dropout(0.25),
    keras.layers.SimpleRNN(32, return_sequences=True),
    keras.layers.Dropout(0.25),
    keras.layers.SimpleRNN(32),
    keras.layers.Dense(future_steps),
    keras.layers.Reshape([future_steps, 1])
])

# Compile the model
model_rnn.compile(loss=tf.keras.losses.Huber(), optimizer="adam", metrics=["mae"])

# Summarize
model_rnn.summary()

# Early Stopping
early_stopping = keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)

# Train
history_rnn = model_rnn.fit(
    X_train, Y_train,
    epochs=epochs,
    validation_data=(X_valid, Y_valid),
    callbacks=[early_stopping],
    verbose=1
)

# Plot Loss and MAE for training and validation
fig, ax1 = plt.subplots(figsize=(10, 5))

# Loss (left axis)
ax1.plot(history_rnn.history["loss"], label="Train Loss")
ax1.plot(history_rnn.history["val_loss"], label="Val Loss", linestyle="--")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss (MSE)")
ax1.grid(True)

# MAE (right axis)
ax2 = ax1.twinx()
ax2.plot(history_rnn.history["mae"], label="Train MAE", linestyle=":")
ax2.plot(history_rnn.history["val_mae"], label="Val MAE", linestyle="-.")
ax2.set_ylabel("MAE")

# Title
plt.title("Training vs Validation Metrics ")

# Combined legend
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()

ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right")

plt.show()

# Predict for the test set
Y_pred_rnn = model_rnn.predict(X_test)

# Plot the first six samples along with corresponding targets and predictions
for i in range(6):
    print(f"Example {i}")
    plot_series(
        inverse_transform(X_test[i],scaler),
        inverse_transform(Y_test[i],scaler),
        inverse_transform(Y_pred_rnn[i],scaler)
    )

# MAE and RMSE for RNN
mae_rnn = mean_absolute_error(
    inverse_transform(Y_test.reshape(-1),scaler),
    inverse_transform(Y_pred_rnn.reshape(-1),scaler)
)

rmse_rnn = root_mean_squared_error(
    inverse_transform(Y_test.reshape(-1),scaler),
    inverse_transform(Y_pred_rnn.reshape(-1),scaler)
)

# Print MAE and RMSE
print("RNN MAE:", mae_rnn)
print("RNN RMSE:", rmse_rnn)

model_results.append({
    "Model": "RNN",
    "MAE": mae_rnn,
    "RMSE": rmse_rnn
})


In [ ]:
# Function what creates windows
def to_windows(dataset, length):
    dataset = dataset.window(length, shift=1, drop_remainder=True)
    return dataset.flat_map(lambda window_ds: window_ds.batch(length))

# Function that creates the tensor form for training
def to_seq2seq_dataset(series, seq_length=past_steps, ahead=future_steps, target_col=0, batch_size=16, shuffle=False, seed=42):
    ds = to_windows(tf.data.Dataset.from_tensor_slices(series), ahead + 1)
    ds = to_windows(ds, seq_length).map(lambda S: (S[:, 0, :], S[:, 1:, target_col]))
    if shuffle:
        ds = ds.shuffle(8 * batch_size, seed=seed)
    return ds.batch(batch_size)

train_series_scaled_seq = to_seq2seq_dataset(train_series_scaled.reshape(-1, 1))
valid_series_scaled_seq = to_seq2seq_dataset(valid_series_scaled.reshape(-1, 1))
test_series_scaled_seq = to_seq2seq_dataset(test_series_scaled.reshape(-1, 1))

# Create default GRU
model_gru = keras.models.Sequential([
    keras.layers.Input(shape=[past_steps, 1]),
    keras.layers.GRU(32, return_sequences=True),
    keras.layers.Dropout(0.25),
    keras.layers.GRU(32, return_sequences=True),
    keras.layers.Dropout(0.25),
    keras.layers.GRU(32, return_sequences=True),
    keras.layers.Dense(future_steps)
])

# Compile the model
model_gru.compile(loss=tf.keras.losses.Huber(), optimizer="adam", metrics=["mae"])

# Summarize
model_gru.summary()

# Early Stopping
early_stopping = keras.callbacks.EarlyStopping(
    patience=5,
    restore_best_weights=True
)

# Train
history_gru = model_gru.fit(
    train_series_scaled_seq,
    epochs=epochs,
    validation_data=valid_series_scaled_seq,
    callbacks=[early_stopping],
    verbose=1
)

# Plot Loss and MAE for training and validation
fig, ax1 = plt.subplots(figsize=(10, 5))

# Loss (left axis)
ax1.plot(history_gru.history["loss"], label="Train Loss")
ax1.plot(history_gru.history["val_loss"], label="Val Loss", linestyle="--")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss (MSE)")
ax1.grid(True)

# MAE (right axis)
ax2 = ax1.twinx()
ax2.plot(history_gru.history["mae"], label="Train MAE", linestyle=":")
ax2.plot(history_gru.history["val_mae"], label="Val MAE", linestyle="-.")
ax2.set_ylabel("MAE")

# Title
plt.title("Training vs Validation Metrics ")

# Combined legend
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()

ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right")

plt.show()

# Predict for the test set
Y_pred_gru = model_gru.predict(X_test)

# Keep only the prediction at the last input time step
Y_pred_gru = Y_pred_gru[:, -1, :]          # shape: (n_samples, future_steps)

# Reshape to match Y_test: (n_samples, future_steps, 1)
Y_pred_gru = Y_pred_gru[..., np.newaxis]

# Plot the first six samples along with corresponding targets and predictions
for i in range(6):
    print(f"Example {i}")
    plot_series(
        inverse_transform(X_test[i],scaler),
        inverse_transform(Y_test[i],scaler),
        inverse_transform(Y_pred_gru[i],scaler)
    )

# MAE and RMSE for RNN
mae_gru = mean_absolute_error(
    inverse_transform(Y_test.reshape(-1),scaler),
    inverse_transform(Y_pred_gru.reshape(-1),scaler)
)

rmse_gru = root_mean_squared_error(
    inverse_transform(Y_test.reshape(-1),scaler),
    inverse_transform(Y_pred_gru.reshape(-1),scaler)
)

# Print MAE and RMSE
print("GRU MAE:", mae_gru)
print("GRU RMSE:", rmse_gru)

model_results.append({
    "Model": "GRU",
    "MAE": mae_gru,
    "RMSE": rmse_gru
})


In [ ]:

# naive forecast: tomorrow = today
naive_pred = test_series[:-1]
naive_true = test_series[1:]

naive_mae = mean_absolute_error(naive_true, naive_pred)
naive_rmse = np.sqrt(mean_squared_error(naive_true, naive_pred))

print("Naive MAE:", naive_mae)
print("Naive RMSE:", naive_rmse)

model_results.append({
    "Model": "Naive",
    "MAE": naive_mae,
    "RMSE": naive_rmse
})

In [ ]:
#Calculate Arima and Sarima models as baseline to compare with the results of the NNs
arima_model = ARIMA(train_series, order=(1, 1, 1))
arima_fit = arima_model.fit()
arima_pred = arima_fit.forecast(steps=len(test_series))

arima_mae = mean_absolute_error(test_series, arima_pred)
arima_rmse = np.sqrt(mean_squared_error(test_series, arima_pred))

print(f"ARIMA MAE: {arima_mae:.2f}")
print(f"ARIMA RMSE: {arima_rmse:.2f}")

model_results.append({
    "Model": "ARIMA",
    "MAE": arima_mae,
    "RMSE": arima_rmse
})


# Set s=7 if keeping a natural week, or s=6 if you dropped Sundays.
sarima_model = SARIMAX(
    train_series, 
    order=(1, 1, 1), 
    seasonal_order=(1, 1, 1, 6) 
)
sarima_fit = sarima_model.fit(disp=False)
sarima_pred = sarima_fit.forecast(steps=len(test_series))

sarima_mae = mean_absolute_error(test_series, sarima_pred)
sarima_rmse = np.sqrt(mean_squared_error(test_series, sarima_pred))

print(f"SARIMA MAE: {sarima_mae:.2f}")
print(f"SARIMA RMSE: {sarima_rmse:.2f}")

model_results.append({
    "Model": "SARIMA",
    "MAE": sarima_mae,
    "RMSE": sarima_rmse
})

I will now transform the data by adding all Sundays and with 3 new columns indicating if its Sundau, Monday and Saturday. Reasoning is that this way the model will predict 7 days instead of 6 while changing the window to 28 from 26. Also since the e-shop is usually closed Sundays and always Saturday is open half the normal hours I want the model to capture that . As a result Monday has higher daily sales than the other days and Saturday has less than Friday.

In [ ]:
# create full daily date range
full_index = pd.date_range(
    start=data.index.min(),
    end=data.index.max(),
    freq="D"
)

# add missing days
data = data.reindex(full_index)

# fill missing sales with 0
data["sales"] = data["sales"].fillna(0)

# date features
data["is_monday"] = (data.index.weekday == 0).astype(np.float32)
data["is_saturday"] = (data.index.weekday == 5).astype(np.float32)
data["is_sunday"] = (data.index.weekday == 6).astype(np.float32)

In [ ]:
past_steps = 28
future_steps = 7
batch_size = 16
epochs = 200

In [ ]:
sales_values = data["sales"].values.astype(np.float32)

n = len(sales_values)
train_end = int(n * 0.70)
valid_end = int(n * 0.85)

train_sales = sales_values[:train_end]
valid_sales = sales_values[train_end:valid_end]
test_sales = sales_values[valid_end:]

scaler = StandardScaler()
scaler.fit(train_sales.reshape(-1, 1))

train_sales_scaled = scaler.transform(train_sales.reshape(-1, 1)).flatten()
valid_sales_scaled = scaler.transform(valid_sales.reshape(-1, 1)).flatten()
test_sales_scaled = scaler.transform(test_sales.reshape(-1, 1)).flatten()

In [ ]:
features = data[["is_monday", "is_saturday", "is_sunday"]].values.astype(np.float32)

train_features = features[:train_end]
valid_features = features[train_end:valid_end]
test_features = features[valid_end:]

train_series_scaled = np.column_stack([train_sales_scaled, train_features])
valid_series_scaled = np.column_stack([valid_sales_scaled, valid_features])
test_series_scaled = np.column_stack([test_sales_scaled, test_features])

In [ ]:
train_series_scaled_seq = to_seq2seq_dataset(
    train_series_scaled,
    seq_length=past_steps,
    ahead=future_steps,
    batch_size=batch_size,
    shuffle=True
)

valid_series_scaled_seq = to_seq2seq_dataset(
    valid_series_scaled,
    seq_length=past_steps,
    ahead=future_steps,
    batch_size=batch_size
)

test_series_scaled_seq = to_seq2seq_dataset(
    test_series_scaled,
    seq_length=past_steps,
    ahead=future_steps,
    batch_size=batch_size
)

In [ ]:
for X_batch, Y_batch in train_series_scaled_seq.take(1):
    print("X_batch:", X_batch.shape)
    print("Y_batch:", Y_batch.shape)

In [ ]:
def dataset_to_numpy(ds):
    X_list = []
    Y_list = []

    for X_batch, Y_batch in ds:
        X_list.append(X_batch.numpy())
        Y_list.append(Y_batch.numpy())

    X = np.concatenate(X_list, axis=0)
    Y = np.concatenate(Y_list, axis=0)

    return X, Y

In [ ]:
X_train, Y_train = dataset_to_numpy(train_series_scaled_seq)
X_valid, Y_valid = dataset_to_numpy(valid_series_scaled_seq)
X_test, Y_test = dataset_to_numpy(test_series_scaled_seq)

In [ ]:
#New GRU model with 4 inputs instead of 1
new_gru = keras.models.Sequential([
    keras.layers.Input(shape=[past_steps, 4]),
    keras.layers.GRU(32, return_sequences=True),
    keras.layers.GRU(32, return_sequences=True),
    keras.layers.GRU(32, return_sequences=True),
    keras.layers.Dense(future_steps)
])

# Compile the model
new_gru.compile(loss=tf.keras.losses.Huber(), optimizer="adam", metrics=["mae"])

# Summarize
new_gru.summary()

# Early Stopping
early_stopping = keras.callbacks.EarlyStopping(
    patience=5,
    restore_best_weights=True
)

# Train
history_new_gru = new_gru.fit(
    train_series_scaled_seq,
    epochs=epochs,
    validation_data=valid_series_scaled_seq,
    callbacks=[early_stopping],
    verbose=1
)


In [ ]:
# Plot Loss and MAE for training and validation
fig, ax1 = plt.subplots(figsize=(10, 5))

# Loss (left axis)
ax1.plot(history_new_gru .history["loss"], label="Train Loss")
ax1.plot(history_new_gru .history["val_loss"], label="Val Loss", linestyle="--")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss (MSE)")
ax1.grid(True)

# MAE (right axis)
ax2 = ax1.twinx()
ax2.plot(history_new_gru .history["mae"], label="Train MAE", linestyle=":")
ax2.plot(history_new_gru .history["val_mae"], label="Val MAE", linestyle="-.")
ax2.set_ylabel("MAE")

# Title
plt.title("Training vs Validation Metrics ")

# Combined legend
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()

ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right")

plt.show()

# Predict
Y_pred_new_gru = new_gru.predict(X_test)

print("Raw Y_test shape:", Y_test.shape)
print("Raw Y_pred_new_gru shape:", Y_pred_new_gru.shape)



In [ ]:
# Keep only last timestep forecast
Y_pred_new_gru = Y_pred_new_gru[:, -1, :]              # (samples, future_steps)
Y_pred_new_gru = Y_pred_new_gru[..., np.newaxis]       # (samples, future_steps, 1)

# Keep only last timestep target
Y_test_last = Y_test[:, -1, :]                         # (samples, future_steps)
Y_test_last = Y_test_last[..., np.newaxis]             # (samples, future_steps, 1)

print("Final Y_test_last shape:", Y_test_last.shape)
print("Final Y_pred_new_gru shape:", Y_pred_new_gru.shape)

mae_new_gru = mean_absolute_error(
    inverse_transform(Y_test_last.reshape(-1), scaler),
    inverse_transform(Y_pred_new_gru.reshape(-1), scaler)
)

rmse_new_gru = root_mean_squared_error(
    inverse_transform(Y_test_last.reshape(-1), scaler),
    inverse_transform(Y_pred_new_gru.reshape(-1), scaler)
)

print("NEW GRU MAE:", mae_new_gru)
print("NEW GRU RMSE:", rmse_new_gru)

for i in range(6):
    print(f"Example {i}")

    plot_series(
        inverse_transform(X_test[i, :, 0].reshape(-1, 1), scaler),
        inverse_transform(Y_test_last[i].reshape(-1, 1), scaler),
        inverse_transform(Y_pred_new_gru[i].reshape(-1, 1), scaler)
    )

In [ ]:
model_results.append({
    "Model": "NEW GRU",
    "MAE": float(mae_new_gru),
    "RMSE": float(rmse_new_gru)
})

In [ ]:
results_df = pd.DataFrame(model_results)
results_df.sort_values("MAE")

In [ ]:
print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)
print("X_valid:", X_valid.shape)
print("Y_valid:", Y_valid.shape)
print("X_test:", X_test.shape)
print("Y_test:", Y_test.shape)

In [ ]:
train_dates = data.index[:train_end]
valid_dates = data.index[train_end:valid_end]
test_dates = data.index[valid_end:]

print("Train:", train_dates.min(), "to", train_dates.max())
print("Valid:", valid_dates.min(), "to", valid_dates.max())
print("Test :", test_dates.min(), "to", test_dates.max())

print(train_dates.max() < valid_dates.min())
print(valid_dates.max() < test_dates.min())

In [ ]:
Y_test_eval = Y_test_last[::future_steps]
Y_pred_eval = Y_pred_new_gru[::future_steps]

mae_strict = mean_absolute_error(
    inverse_transform(Y_test_eval.reshape(-1), scaler),
    inverse_transform(Y_pred_eval.reshape(-1), scaler)
)

rmse_strict = root_mean_squared_error(
    inverse_transform(Y_test_eval.reshape(-1), scaler),
    inverse_transform(Y_pred_eval.reshape(-1), scaler)
)

print("Strict non-overlap MAE:", mae_strict)
print("Strict non-overlap RMSE:", rmse_strict)

In [ ]:
model_results.append({
    "Model": "NEW GRU 28/7 + Weekday Flags Strict",
    "MAE": float(mae_strict),
    "RMSE": float(rmse_strict)
})

Now that we have our models I will use actual data for prediction and make comparisons. I have seperated the data from 26/6 till 22/8 and I will use each week to make Predicitions without giving the actual data to the model. Then I will compare the results , add the week to the data retrain and make a new prediction for the next week again without giving the results to the model.

In [ ]:
# Actual sales for real walk-forward evaluation
actual_sales = data2.loc["2026-06-29":"2026-08-22"].copy()

# Keep only Monday-Saturday
actual_sales = actual_sales[actual_sales.index.weekday != 6].copy()

# Keep only target column
actual_sales = actual_sales[["sales"]].copy()

# Add useful evaluation columns
actual_sales["forecast_date"] = actual_sales.index
actual_sales["week_start"] = actual_sales["forecast_date"] - pd.to_timedelta(
    actual_sales["forecast_date"].dt.weekday, unit="D"
)

actual_sales["horizon"] = actual_sales.groupby("week_start").cumcount() + 1

# Clean column names
actual_sales = actual_sales.rename(columns={"sales": "y_true"})

# Final shape
actual_sales = actual_sales.reset_index(drop=True)

actual_sales.head(10)

In [ ]:
actual_sales.groupby("week_start").size()

In [ ]:
past_steps = 26
future_steps = 6
target_col = "sales"

def scale_current_history(current_history, target_col="sales"):
    scaler = StandardScaler()

    sales = current_history[target_col].values.astype(np.float32)
    sales_scaled = scaler.fit_transform(sales.reshape(-1, 1)).astype(np.float32)

    return sales_scaled, scaler


In [ ]:
def prepare_base_model_data(
    current_history,
    target_col="sales",
    past_steps=26,
    future_steps=6,
    batch_size=16
):
    sales_scaled, scaler = scale_current_history(current_history, target_col=target_col)

    # RNN: seq-to-vector
    X_rnn, Y_rnn = make_windows(
        sales_scaled.reshape(-1),
        past_steps=past_steps,
        future_steps=future_steps
    )

    # GRU: seq-to-seq
    gru_ds = to_seq2seq_dataset(
        sales_scaled,
        seq_length=past_steps,
        ahead=future_steps,
        target_col=0,
        batch_size=batch_size,
        shuffle=False,
        seed=42
    )

    # Last known window for next-week prediction
    X_next = sales_scaled[-past_steps:].reshape(1, past_steps, 1).astype(np.float32)

    return {
        "sales_scaled": sales_scaled,
        "scaler": scaler,
        "X_rnn": X_rnn,
        "Y_rnn": Y_rnn,
        "gru_ds": gru_ds,
        "X_next": X_next
    }

In [ ]:
holdout_start = pd.Timestamp("2026-06-29")

current_history = data2.loc[data2.index < holdout_start].copy()
base_data = prepare_base_model_data(
    current_history,
    target_col="sales",
    past_steps=past_steps,
    future_steps=future_steps,
    batch_size=batch_size
)

In [ ]:
print(current_history.index.min())
print(current_history.index.max())

In [ ]:
print("RNN X:", base_data["X_rnn"].shape)
print("RNN Y:", base_data["Y_rnn"].shape)
print("Next window:", base_data["X_next"].shape)

for X_batch, Y_batch in base_data["gru_ds"].take(1):
    print("GRU X batch:", X_batch.shape)
    print("GRU Y batch:", Y_batch.shape)

In [ ]:
def split_windows_time_order(X, Y, valid_ratio=0.15):
    valid_size = int(len(X) * valid_ratio)

    X_train = X[:-valid_size]
    Y_train = Y[:-valid_size]

    X_valid = X[-valid_size:]
    Y_valid = Y[-valid_size:]

    return X_train, Y_train, X_valid, Y_valid

In [ ]:
def build_base_rnn(past_steps=26, future_steps=6):
    model_rnn = keras.models.Sequential([
        keras.layers.Input(shape=[past_steps, 1]),
        keras.layers.SimpleRNN(32, return_sequences=True),
        keras.layers.Dropout(0.25),
        keras.layers.SimpleRNN(32, return_sequences=True),
        keras.layers.Dropout(0.25),
        keras.layers.SimpleRNN(32),
        keras.layers.Dense(future_steps),
        keras.layers.Reshape([future_steps, 1])
    ])

    model_rnn.compile(
        loss=tf.keras.losses.Huber(),
        optimizer="adam",
        metrics=["mae"]
    )

    return model_rnn

In [ ]:
def train_predict_rnn(
    current_history,
    target_col="sales",
    past_steps=26,
    future_steps=6,
    epochs=200,
    batch_size=16
):
    base_data = prepare_base_model_data(
        current_history=current_history,
        target_col=target_col,
        past_steps=past_steps,
        future_steps=future_steps,
        batch_size=batch_size
    )

    X_rnn = base_data["X_rnn"]
    Y_rnn = base_data["Y_rnn"]
    X_next = base_data["X_next"]
    scaler = base_data["scaler"]

    X_train, Y_train, X_valid, Y_valid = split_windows_time_order(X_rnn, Y_rnn)

    model_rnn = build_base_rnn(
        past_steps=past_steps,
        future_steps=future_steps
    )

    early_stopping = keras.callbacks.EarlyStopping(
        patience=5,
        restore_best_weights=True
    )

    history_rnn = model_rnn.fit(
        X_train,
        Y_train,
        epochs=epochs,
        validation_data=(X_valid, Y_valid),
        callbacks=[early_stopping],
        batch_size=batch_size,
        verbose=1
    )

    y_pred_scaled = model_rnn.predict(X_next)
    y_pred = scaler.inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    ).reshape(-1)

    return y_pred, model_rnn, history_rnn

In [ ]:
rnn_results = []

week_starts = sorted(actual_sales["week_start"].unique())

for week_start in week_starts:
    print(f"Training RNN for week starting {week_start.date()}")

    # Only past data is allowed
    current_history = data2.loc[data2.index < week_start].copy()

    # Actual answers for this week
    actual_week = actual_sales[actual_sales["week_start"] == week_start].copy()

    # Safety check
    if len(actual_week) != future_steps:
        print(f"Skipping {week_start.date()} - expected {future_steps} days, got {len(actual_week)}")
        continue

    # Train and predict
    y_pred, model_rnn, history_rnn = train_predict_rnn(
        current_history=current_history,
        target_col="sales",
        past_steps=past_steps,
        future_steps=future_steps,
        epochs=epochs,
        batch_size=batch_size
    )
    print("Predict week:", week_start.date())
    print("History ends:", current_history.index.max().date())
    print("Actual week rows:", len(actual_week))
    # Save prediction table
    week_result = actual_week.copy()
    week_result["model"] = "RNN"
    week_result["y_pred"] = y_pred

    week_result = week_result[
        ["week_start", "forecast_date", "horizon", "model", "y_true", "y_pred"]
    ]

    rnn_results.append(week_result)

In [ ]:
rnn_results_df = pd.concat(rnn_results, ignore_index=True)

rnn_results_df.head()

In [ ]:
mae_rnn_walkforward = mean_absolute_error(
    rnn_results_df["y_true"],
    rnn_results_df["y_pred"]
)

rmse_rnn_walkforward = root_mean_squared_error(
    rnn_results_df["y_true"],
    rnn_results_df["y_pred"]
)

print("RNN Walk-forward MAE:", mae_rnn_walkforward)
print("RNN Walk-forward RMSE:", rmse_rnn_walkforward)

In [ ]:
rnn_weekly_metrics = (
    rnn_results_df
    .groupby("week_start")
    .apply(lambda df: pd.Series({
        "MAE": mean_absolute_error(df["y_true"], df["y_pred"]),
        "RMSE": root_mean_squared_error(df["y_true"], df["y_pred"])
    }))
    .reset_index()
)

rnn_weekly_metrics

In [ ]:
all_model_results = []

all_model_results.append(rnn_results_df)

In [ ]:
def build_base_gru(past_steps=26, future_steps=6):
    model_gru = keras.models.Sequential([
        keras.layers.Input(shape=[past_steps, 1]),
        keras.layers.GRU(32, return_sequences=True),
        keras.layers.Dropout(0.25),
        keras.layers.GRU(32, return_sequences=True),
        keras.layers.Dropout(0.25),
        keras.layers.GRU(32, return_sequences=True),
        keras.layers.Dense(future_steps)
    ])

    model_gru.compile(
        loss=tf.keras.losses.Huber(),
        optimizer="adam",
        metrics=["mae"]
    )

    return model_gru

In [ ]:
def train_predict_gru(
    current_history,
    target_col="sales",
    past_steps=26,
    future_steps=6,
    epochs=200,
    batch_size=16
):
    sales_scaled, scaler = scale_current_history(
        current_history,
        target_col=target_col
    )

    # Create seq-to-seq dataset, no shuffle yet
    gru_ds_all = to_seq2seq_dataset(
        sales_scaled,
        seq_length=past_steps,
        ahead=future_steps,
        target_col=0,
        batch_size=batch_size,
        shuffle=False
    )

    X_gru, Y_gru = dataset_to_numpy(gru_ds_all)

    X_train, Y_train, X_valid, Y_valid = split_windows_time_order(
        X_gru,
        Y_gru
    )

    model_gru = build_base_gru(
        past_steps=past_steps,
        future_steps=future_steps
    )

    early_stopping = keras.callbacks.EarlyStopping(
        patience=5,
        restore_best_weights=True
    )

    history_gru = model_gru.fit(
        X_train,
        Y_train,
        epochs=epochs,
        validation_data=(X_valid, Y_valid),
        callbacks=[early_stopping],
        batch_size=batch_size,
        shuffle=True,
        verbose=1
    )

    # Last known 26 days
    X_next = sales_scaled[-past_steps:].reshape(1, past_steps, 1).astype(np.float32)

    # Seq-to-seq prediction
    y_pred_scaled = model_gru.predict(X_next)

    # Take only last time step
    y_pred_scaled = y_pred_scaled[:, -1, :]      # shape: (1, 6)

    # Back to real scale
    y_pred = scaler.inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    ).reshape(-1)

    return y_pred, model_gru, history_gru

In [ ]:
gru_results = []

week_starts = sorted(actual_sales["week_start"].unique())

for week_start in week_starts:
    print(f"Training GRU for week starting {week_start.date()}")

    # only past data
    current_history = data2.loc[data2.index < week_start].copy()

    # real answers for this week
    actual_week = actual_sales[actual_sales["week_start"] == week_start].copy()

    if len(actual_week) != future_steps:
        print(f"Skipping {week_start.date()} - expected {future_steps} days, got {len(actual_week)}")
        continue

    y_pred, model_gru, history_gru = train_predict_gru(
        current_history=current_history,
        target_col="sales",
        past_steps=past_steps,
        future_steps=future_steps,
        epochs=epochs,
        batch_size=batch_size
    )

    week_result = actual_week.copy()
    week_result["model"] = "GRU"
    week_result["y_pred"] = y_pred

    week_result = week_result[
        ["week_start", "forecast_date", "horizon", "model", "y_true", "y_pred"]
    ]

    gru_results.append(week_result)

In [ ]:
gru_results_df = pd.concat(gru_results, ignore_index=True)

gru_results_df.head()

In [ ]:
mae_gru_walkforward = mean_absolute_error(
    gru_results_df["y_true"],
    gru_results_df["y_pred"]
)

rmse_gru_walkforward = root_mean_squared_error(
    gru_results_df["y_true"],
    gru_results_df["y_pred"]
)

print("GRU Walk-forward MAE:", mae_gru_walkforward)
print("GRU Walk-forward RMSE:", rmse_gru_walkforward)